In [1]:
from typing import Tuple, List, Union, Callable, Any
from contextlib import nullcontext
from itertools import repeat
from collections import UserDict

import torch

from torch import nn, Tensor
from torch.amp import GradScaler, autocast
from torch.utils.checkpoint import get_device_states, set_device_states

class RandContext:
    def __init__(self, *tensors):
        self.fwd_cpu_state = torch.get_rng_state()
        self.fwd_gpu_devices, self.fwd_gpu_states = get_device_states(*tensors)

    def __enter__(self):
        self._fork = torch.random.fork_rng(
            devices=self.fwd_gpu_devices,
            enabled=True
        )
        self._fork.__enter__()
        torch.set_rng_state(self.fwd_cpu_state)
        set_device_states(self.fwd_gpu_devices, self.fwd_gpu_states)

    def __exit__(self, exc_type, exc_val, exc_tb):
        self._fork.__exit__(exc_type, exc_val, exc_tb)
        self._fork = None

class GradCache:
    def __init__(
            self,
            models: List[nn.Module],
            chunk_sizes: Union[int, List[int]],
            loss_fn: Callable[..., Tensor],
            split_input_fn: Callable[[Any, int], Any] = None,
            get_rep_fn: Callable[..., Tensor] = None,
            fp16: bool = False,
            scaler: GradScaler = None,
    ):
        self.models = models

        if isinstance(chunk_sizes, int):
            self.chunk_sizes = [chunk_sizes for _ in range(len(models))]
        else:
            self.chunk_sizes = chunk_sizes

        self.split_input_fn = split_input_fn
        self.get_rep_fn = get_rep_fn
        self.loss_fn = loss_fn

        if fp16:
            assert scaler is not None, "mixed precision training requires a gradient scaler passed in"

        self.fp16 = fp16
        self.scaler = scaler

        self._get_input_tensors_strict = False

    def __call__(self, *args, **kwargs):
        return self.cache_step(*args, **kwargs)

    def split_inputs(self, model_input, chunk_size: int) -> List:
        # delegate splitting to user provided function
        if self.split_input_fn is not None:
            return self.split_input_fn(model_input, chunk_size)

        if isinstance(model_input, (dict, UserDict)) and all(isinstance(x, Tensor) for x in model_input.values()):
            keys = list(model_input.keys())
            chunked_tensors = [model_input[k].split(chunk_size, dim=0) for k in keys]
            return [dict(zip(kk, tt)) for kk, tt in zip(repeat(keys), zip(*chunked_tensors))]

        elif isinstance(model_input, list) and all(isinstance(x, Tensor) for x in model_input):
            chunked_x = [t.split(chunk_size, dim=0) for t in model_input]
            return [list(s) for s in zip(*chunked_x)]

        elif isinstance(model_input, Tensor):
            return list(model_input.split(chunk_size, dim=0))

        elif isinstance(model_input, tuple) and list(map(type, model_input)) == [list, dict]:
            args_chunks = self.split_inputs(model_input[0], chunk_size)
            kwargs_chunks = self.split_inputs(model_input[1], chunk_size)
            return list(zip(args_chunks, kwargs_chunks))

        else:
            raise NotImplementedError(f'Model input split not implemented for type {type(model_input)}')

    def get_input_tensors(self, model_input) -> List[Tensor]:
        if isinstance(model_input, Tensor):
            return [model_input]

        elif isinstance(model_input, (list, tuple)):
            return sum((self.get_input_tensors(x) for x in model_input), [])

        elif isinstance(model_input, (dict, UserDict)):
            return sum((self.get_input_tensors(x) for x in model_input.values()), [])

        elif self._get_input_tensors_strict:
            raise NotImplementedError(f'get_input_tensors not implemented for type {type(model_input)}')

        else:
            return []

    def model_call(self, model: nn.Module, model_input):
        with autocast('cuda') if self.fp16 else nullcontext():
            if isinstance(model_input, Tensor):
                return model(model_input)
            elif isinstance(model_input, list):
                return model(*model_input)
            elif isinstance(model_input, (dict, UserDict)):
                return model(**model_input)
            elif isinstance(model_input, tuple) and list(map(type, model_input)) == [list, dict]:
                model_args, model_kwargs = model_input
                return model(*model_args, **model_kwargs)
            else:
                raise NotImplementedError

    def get_reps(self, model_out) -> Tensor:
        if self.get_rep_fn is not None:
            return self.get_rep_fn(model_out)
        else:
            return model_out

    def compute_loss(self, *reps: Tensor, **loss_kwargs) -> Tensor:
        loss = self.loss_fn(*reps, **loss_kwargs)
        return loss

    def forward_no_grad(
            self,
            model: nn.Module,
            model_inputs,
    ) -> [Tensor, List[RandContext]]:
        rnd_states = []
        model_reps = []

        with torch.no_grad():
            for x in model_inputs:
                rnd_states.append(RandContext(*self.get_input_tensors(x)))
                y = self.model_call(model, x)
                model_reps.append(self.get_reps(y))

        # concatenate all sub-batch representations
        model_reps = torch.cat(model_reps, dim=0)
        return model_reps, rnd_states

    def build_cache(self, *reps: Tensor, **loss_kwargs) -> [List[Tensor], Tensor]:
        reps = [r.detach().requires_grad_() for r in reps]
        with autocast('cuda') if self.fp16 else nullcontext():
            loss = self.compute_loss(*reps, **loss_kwargs)

        if self.fp16:
            self.scaler.scale(loss).backward()
        else:
            loss.backward()

        cache = [r.grad for r in reps]

        return cache, loss.detach()

    def forward_backward(
            self,
            model: nn.Module,
            model_inputs,
            cached_gradients: List[Tensor],
            random_states: List[RandContext],
            no_sync_except_last: bool = False
    ):
        if no_sync_except_last:
            sync_contexts = [model.no_sync for _ in range(len(model_inputs) - 1)] + [nullcontext]
        else:
            sync_contexts = [nullcontext for _ in range(len(model_inputs))]

        for x, state, gradient, sync_context in zip(model_inputs, random_states, cached_gradients, sync_contexts):
            with sync_context():
                with state:
                    y = self.model_call(model, x)
                reps = self.get_reps(y)

                surrogate = torch.dot(reps.flatten(), gradient.flatten())
                surrogate.backward()

    def cache_step(
            self,
            *model_inputs,
            no_sync_except_last: bool = False,
            **loss_kwargs
    ) -> Tensor:
        all_reps = []
        all_rnd_states = []

        if no_sync_except_last:
            assert all(map(lambda m: isinstance(m, nn.parallel.DistributedDataParallel), self.models)), \
                'Some of models are not wrapped in DistributedDataParallel. Make sure you are running DDP with ' \
                'proper initializations.'

        model_inputs = [self.split_inputs(x, chunk_size) for x, chunk_size in zip(model_inputs, self.chunk_sizes)]

        for model, x in zip(self.models, model_inputs):
            model_reps, rnd_states = self.forward_no_grad(model, x)
            all_reps.append(model_reps)
            all_rnd_states.append(rnd_states)

        cache, loss = self.build_cache(*all_reps, **loss_kwargs)
        cache = [c.split(chunk_size) for c, chunk_size in zip(cache, self.chunk_sizes)]

        for model, x, model_cache, rnd_states in zip(
                self.models, model_inputs, cache, all_rnd_states):
            self.forward_backward(model, x, model_cache, rnd_states, no_sync_except_last=no_sync_except_last)

        return loss


class PLGradCache(GradCache):
    def __init__(
        self,
        models: List[nn.Module],
        chunk_sizes: Union[int, List[int]],
        loss_fn: Callable[..., Tensor],
        split_input_fn: Callable[[Any, int], Any] = None,
        get_rep_fn: Callable[..., Tensor] = None,
        fp16: bool = False,
        scaler: GradScaler = None,
        backward_fn=None,  # [added]
    ):
        super().__init__(models, chunk_sizes, loss_fn, split_input_fn, get_rep_fn, fp16, scaler)
        self.backward_fn = backward_fn

    def build_cache(self, *reps: Tensor, **loss_kwargs) -> Union[List[Tensor], Tensor]:
        reps = [r.detach().requires_grad_() for r in reps]
        with autocast('cuda') if self.fp16 else nullcontext():
            loss = self.compute_loss(*reps, **loss_kwargs)

        self.backward_fn(loss)  # [modified]

        cache = [r.grad for r in reps]

        return cache, loss.detach()

    def forward_backward(
        self,
        model: nn.Module,
        model_inputs,
        cached_gradients: List[Tensor],
        random_states: List[RandContext],
        no_sync_except_last: bool = False,
    ):
        if isinstance(
            model, nn.parallel.DistributedDataParallel
        ):  # [use ddp_model]

            if no_sync_except_last:
                sync_contexts = [
                    model.no_sync for _ in range(len(model_inputs) - 1)
                ] + [nullcontext]
                sync_flags = [True] * (len(model_inputs))  # [added]
            else:
                sync_contexts = [nullcontext for _ in range(len(model_inputs))]
                sync_flags = [False] * (len(model_inputs))  # [added]

            # [modified]
            for x, state, gradient, sync_context, sync_flag in zip(
                model_inputs, random_states, cached_gradients, sync_contexts, sync_flags
            ):
                with sync_context():
                    with state:
                        y = self.model_call(model, x)
                    reps = self.get_reps(y)
                    surrogate = torch.dot(reps.flatten(), gradient.flatten())
                    if sync_flag:
                        model.require_backward_grad_sync = True
                    if self.fp16:  # [added]
                        self.scaler._enabled = False
                        self.backward_fn(surrogate)
                        self.scaler._enabled = True
                    else:
                        self.backward_fn(surrogate)  # [modified]
        else:  # [use base model (i.e. SimpleLitModel)]

            # [remove no_sync_except_last: pytorch lightning would handle gradient sync automatically]
            for x, state, gradient in zip(
                model_inputs, random_states, cached_gradients
            ):
                with state:
                    y = self.model_call(model, x)
                reps = self.get_reps(y)
                surrogate = torch.dot(reps.flatten(), gradient.flatten())
                if self.fp16:  # [added]
                    self.scaler._enabled = False
                    self.backward_fn(surrogate)
                    self.scaler._enabled = True
                else:
                    self.backward_fn(surrogate)  # [added]

    def cache_step(
        self, *model_inputs, no_sync_except_last: bool = False, **loss_kwargs
    ) -> Tuple[Tensor, Tensor]:
        all_reps = []
        all_rnd_states = []

        model_inputs = [
            self.split_inputs(x, chunk_size)
            for x, chunk_size in zip(model_inputs, self.chunk_sizes)
        ]

        for model, x in zip(self.models, model_inputs):
            model_reps, rnd_states = self.forward_no_grad(model, x)
            all_reps.append(model_reps)
            all_rnd_states.append(rnd_states)

        cache, loss = self.build_cache(*all_reps, **loss_kwargs)
        cache = [c.split(chunk_size) for c, chunk_size in zip(cache, self.chunk_sizes)]

        for model, x, model_cache, rnd_states in zip(
            self.models, model_inputs, cache, all_rnd_states
        ):
            self.forward_backward(
                model,
                x,
                model_cache,
                rnd_states,
                no_sync_except_last=no_sync_except_last,
            )

        return loss

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class AsymmetricLearnableInfoNCELoss(nn.Module):
    def __init__(self):
        super(AsymmetricLearnableInfoNCELoss, self).__init__()
        self.w = nn.Parameter(torch.tensor(float(torch.log(torch.tensor(10.0))), requires_grad=True))
        self.b = nn.Parameter(torch.tensor(-10.0, requires_grad=True))

    def forward(self, emb, anchor_emb, indices=None):
        # Assert that the number of embeddings matches if indices are not provided
        if indices is None:
            assert len(emb) == len(anchor_emb), "emb and anchor_emb must have the same length when indices are None."

        # Calculate the similarity matrix
        similarity_matrix = torch.matmul(anchor_emb, emb.T)

        # Apply scaling and bias to the similarity matrix
        logits = torch.exp(self.w) * similarity_matrix + self.b

        # Apply log softmax to logits
        log_probs = F.log_softmax(logits, dim=1)

        # If indices are None, select diagonal elements for the loss
        if indices is None:
            loss = -torch.diagonal(log_probs)  # Diagonal elements represent the self-supervised pairs
            loss = torch.mean(loss)
        else:
            rows, cols = zip(*indices)
            loss = -log_probs[torch.tensor(rows), torch.tensor(cols)]
            loss = torch.sum(loss)/anchor_emb.shape[0]
        
        # Compute mean loss over all samples
        
        return loss


# Define batch size and embedding dimension
batch_size = 4
embedding_dim = 5
num_candidates = 4  # Same as batch_size to check indices == None case

# Random anchor embeddings (batch_size, embedding_dim)
anchor_emb = F.normalize(torch.randn(batch_size, embedding_dim), p=2, dim=1)

# Random candidate embeddings (num_candidates, embedding_dim)
embs = F.normalize(torch.randn(num_candidates, embedding_dim), p=2, dim=1)

# Initialize the custom loss
loss_fn = AsymmetricLearnableInfoNCELoss()

# Case 1: With provided indices
indices = [(0, 2), (0, 1), (1, 1), (2, 3)]
loss = loss_fn(embs, anchor_emb, indices)
print("Computed Asymmetric InfoNCE Loss with indices:", loss.item())

# Case 2: Without provided indices (None)
loss_no_indices = loss_fn(embs, anchor_emb, None)
print("Computed Asymmetric InfoNCE Loss without indices:", loss_no_indices.item())


Computed Asymmetric InfoNCE Loss with indices: 7.490878582000732
Computed Asymmetric InfoNCE Loss without indices: 5.88606071472168


In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class BinarySigmoidLoss(nn.Module):
    def __init__(self):
        super(BinarySigmoidLoss, self).__init__()
        self.w = nn.Parameter(torch.tensor(float(torch.log(torch.tensor(10.0))), requires_grad=True))
        self.b = nn.Parameter(torch.tensor(-10.0, requires_grad=True))

    def forward(self, emb, anchor_emb, indices=None):
        # Assert that the number of embeddings matches if indices are not provided
        if indices is None:
            assert len(emb) == len(anchor_emb), "emb and anchor_emb must have the same length when indices are None."
        
        # Calculate the similarity matrix
        similarity_matrix = torch.matmul(anchor_emb, emb.T)
        
        # Apply scaling and bias to the similarity matrix
        logits = torch.exp(self.w) * similarity_matrix + self.b

        # Create label matrix based on indices if provided
        if indices is not None:
            rows, cols = zip(*indices)
            labels = -torch.ones_like(logits).type_as(logits)
            labels[rows, cols] = 1
            loss = -torch.sum(F.logsigmoid(labels * logits)) / len(indices)
        else:
            # Create default labels: diagonal elements are 1, off-diagonals are -1
            labels = 2 * torch.eye(logits.size(0)).type_as(logits) - torch.ones_like(logits).type_as(logits)
            loss = -torch.sum(F.logsigmoid(labels * logits)) / logits.size(0)
        return loss

# Define batch size and embedding dimension
batch_size = 4
embedding_dim = 5
num_candidates = 4  # Same as batch_size to check indices == None case

# Random anchor embeddings (batch_size, embedding_dim)
anchor_emb = F.normalize(torch.randn(batch_size, embedding_dim), p=2, dim=1)

# Random candidate embeddings (num_candidates, embedding_dim)
embs = F.normalize(torch.randn(num_candidates, embedding_dim), p=2, dim=1)

# Initialize the custom loss
loss_fn = BinarySigmoidLoss()

# Case 1: With provided indices
indices = [(0, 2), (0, 1), (1, 1), (2, 3)]
loss = loss_fn(embs, anchor_emb, indices)
print("Computed BinarySigmoidLoss Loss with indices:", loss.item())

# Case 2: Without provided indices (None)
loss_no_indices = loss_fn(embs, anchor_emb, None)
print("Computed BinarySigmoidLoss Loss without indices:", loss_no_indices.item())

Computed BinarySigmoidLoss Loss with indices: 9.313237190246582
Computed BinarySigmoidLoss Loss without indices: 9.295373916625977


In [4]:
import io
import os
import gc
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import lightning as L

from PIL import Image
from glob import glob
from datasets import load_dataset, concatenate_datasets, load_from_disk
from torch.utils.data import DataLoader
from transformers import AutoImageProcessor, AutoModel, AutoTokenizer
from lightning.pytorch.callbacks import ModelCheckpoint, EarlyStopping
from sklearn.metrics import ndcg_score, recall_score
# from bitsandbytes.optim import AdamW8b

In [5]:
SEED = 42
BATCH_SIZE = 512

image_model_name = "microsoft/dit-base-finetuned-rvlcdip"
text_model_name = 'Alibaba-NLP/gte-base-en-v1.5'

In [6]:
L.seed_everything(SEED)
torch.set_float32_matmul_precision('medium')
Image.MAX_IMAGE_PIXELS = None

Seed set to 42


In [7]:
train_datasets = []
for dataset_path in glob('./preproc/vqa/*'):
    train_datasets.append(load_dataset(dataset_path, split='train'))
train_dataset = concatenate_datasets(train_datasets)

Resolving data files:   0%|          | 0/55 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/27 [00:00<?, ?it/s]

In [8]:
train_dataset

Dataset({
    features: ['image_bytes', 'questions', 'labels'],
    num_rows: 922514
})

In [9]:
# #test for gpu fit
# def preproc(example):
#     example['len'] = -len(example['text'])
#     return example

# train_dataset = train_dataset.map(preproc)
# train_dataset = train_dataset.sort("len")
train_dataset = train_dataset.shuffle()

In [10]:
eval_datasets = []
for dataset_path in glob('./eval_data/*'):
    eval_datasets.append(load_dataset(dataset_path, split='train'))
eval_dataset = concatenate_datasets(eval_datasets)

In [11]:
class TrainCollateFn:
    def __init__(self, image_model_name, text_model_name, device='cuda', gpu_batch=16):
        self.image_processor = AutoImageProcessor.from_pretrained(image_model_name, apply_ocr=False)
        self.text_tokenizer = AutoTokenizer.from_pretrained(text_model_name)
        self.text_model = AutoModel.from_pretrained(text_model_name, trust_remote_code=True)
        self.text_model.eval()  # Ensure the model is in evaluation mode
        self.device = device
        self.gpu_batch = gpu_batch

        # Move the text model to the GPU if specified
        if self.device == 'cuda':
            self.text_model.to(self.device)

    def process_text_batch(self, texts):
        # Tokenize texts and split them into smaller batches to fit into the GPU
        text_inputs = self.text_tokenizer(texts, max_length=8192, padding=True, truncation=True, return_tensors='pt').to(self.device)
        with torch.no_grad():
            text_outputs = self.text_model(**text_inputs)
        
        # Extract CLS token or the first token embedding and normalize
        anchor_emb = text_outputs.last_hidden_state[:, 0]  # Use CLS token or first token
        anchor_emb = F.normalize(anchor_emb, p=2, dim=1)
        return anchor_emb.detach().cpu()

    def __call__(self, examples):
        images = []
        questions = []
        indices = []
        for example in examples:
            labels_row = [(idx + len(questions) , label+len(images))for idx, label in enumerate(example['labels'])]
            indices.extend(labels_row)
            images_row = [Image.open(io.BytesIO(image_bytes)) for image_bytes in example['image_bytes']]
            images.extend(images_row)
            questions.extend(example['questions'])


        # Split texts into batches to process on GPU
        all_anchor_embs = []
        for i in range(0, len(questions), self.gpu_batch):
            batch_questions = questions[i:i + self.gpu_batch]
            anchor_emb = self.process_text_batch(batch_questions)
            all_anchor_embs.append(anchor_emb)

        # Concatenate all embeddings into one tensor
        anchor_emb = torch.cat(all_anchor_embs, dim=0)

        # Preprocess images (this part is often handled on CPU)
        pixel_values = self.image_processor.preprocess(images=images, return_tensors='pt')['pixel_values']

        return {
            'pixel_values': pixel_values,  # Image inputs remain on CPU unless moved explicitly
            'anchor_embs': anchor_emb,     # Anchor embeddings are computed on the GPU if device is 'cuda'
            'indices': indices
        }


In [12]:
class EvalCollateFn:
    def __init__(self, image_model_name, text_model_name, device='cuda'):
        self.image_processor = AutoImageProcessor.from_pretrained(image_model_name, apply_ocr=False)
        self.text_tokenizer = AutoTokenizer.from_pretrained(text_model_name)
        self.text_model = AutoModel.from_pretrained(text_model_name, trust_remote_code=True)
        self.text_model.eval()  # Ensure the model is in evaluation mode
        self.device = device

        # Move the text model to the GPU if specified
        if self.device == 'cuda':
            self.text_model.to(self.device)

    def process_text_batch(self, texts):
        # Tokenize texts and split them into smaller batches to fit into the GPU
        text_inputs = self.text_tokenizer(texts, max_length=8192, padding=True, truncation=True, return_tensors='pt').to(self.device)
        with torch.no_grad():
            text_outputs = self.text_model(**text_inputs)
        
        # Extract CLS token or the first token embedding and normalize
        anchor_emb = text_outputs.last_hidden_state[:, 0]  # Use CLS token or first token
        anchor_emb = F.normalize(anchor_emb, p=2, dim=1)
        return anchor_emb

    def __call__(self, examples):
        example = examples[0]
        images = [Image.open(io.BytesIO(image_byte)) for image_byte in example['image_bytes']]
        pixel_values = self.image_processor.preprocess(images=images, return_tensors='pt')['pixel_values']
        question_embs = self.process_text_batch(example['questions']).cpu().numpy()
       
        return {
            'dataset': example['dataset'],
            'doc_id': example['doc_id'],
            'doc_types':example['doc_types'],
            'pixel_values':pixel_values,
            'question_embs': question_embs,
            'labels': example['labels'],
            'label_types': example['label_types']
        }

    

In [13]:
train_dataloader = DataLoader(train_dataset, collate_fn=TrainCollateFn(image_model_name, text_model_name, device='cuda', gpu_batch=64), shuffle=False, batch_size=BATCH_SIZE, drop_last=True)
eval_dataloader = DataLoader(eval_dataset, collate_fn=EvalCollateFn(image_model_name, text_model_name), batch_size=1)

In [15]:
class LitDocEmbModel(L.LightningModule):
    def __init__(
        self, 
        image_model_name,
        text_model_name,
        mini_batch_size=32
    ):
        super().__init__()
        self.save_hyperparameters()
        self.image_model = AutoModel.from_pretrained(image_model_name)
        self.image_model.train()
        # self.loss = AsymmetricLearnableInfoNCELoss()
        self.loss = BinarySigmoidLoss()
        self.mini_batch_size = mini_batch_size
        self.evaluation_df = pd.DataFrame(columns=['dataset', 'doc_id', 'probs', 'labels', 'doc_type', 'label_types'])
        self.strict_loading = False
        self.automatic_optimization = False

    def init_grad_cache(self, scaler, ddp_module):
        self.trainer.strategy.precision_plugin.forward_context = nullcontext
        self.grad_cache = PLGradCache(
            models=[ddp_module],
            chunk_sizes=self.mini_batch_size,
            loss_fn=self.calculate_loss,
            fp16=True,
            scaler=scaler, # needed when using automatic_optimization is off and fp16 is on
            backward_fn=self.manual_backward, # needed when automatic_optimization is off
        )

    def calculate_loss(self, embs, anchor_embs, indices=None):
        return self.loss(embs, anchor_embs, indices)
        
    def forward(self, pixel_values):
        image_outputs = self.image_model(pixel_values=pixel_values)
        # image_embs = image_outputs.last_hidden_state.mean(dim=1)  # Average the token embeddings to get a single embedding per image
        image_embs = image_outputs.pooler_output
        image_embs = F.normalize(image_embs, p=2, dim=-1)
        return image_embs

    def on_train_start(self): # initialize grad cache here
        self.init_grad_cache(self.trainer.scaler, self.trainer.strategy.model)

    def on_train_epoch_end(self):
        torch.cuda.empty_cache()
        gc.collect()

    def configure_optimizers(self):
        from torch.optim import AdamW
        opt = AdamW(self.parameters(), lr=1e-5)
        return opt
        
    def training_step(self, batch, batch_idx):
        pixel_values, anchor_embs, indices = batch['pixel_values'], batch['anchor_embs'], batch['indices']
        optimizer = self.optimizers()
        optimizer.zero_grad()
        loss = self.grad_cache(
            pixel_values,
            no_sync_except_last=False,
            anchor_embs=anchor_embs,
            indices=indices
        )
        optimizer.step()
        self.log(f"train_loss", loss, on_step=True, on_epoch=False)
        torch.cuda.empty_cache()
        gc.collect()
        return loss
        
    def validation_step(self, batch):
        pixel_values = batch['pixel_values']
        mini_batch_size = self.hparams.mini_batch_size  # Define your mini-batch size

        # Initialize a list to hold embeddings
        image_embs_list = []

        # Process pixel values in mini-batches
        for i in range(0, pixel_values.size(0), mini_batch_size):
            mini_batch = pixel_values[i:i + mini_batch_size]
            with torch.no_grad():  # No need to track gradients
                image_outputs = self.image_model(pixel_values=mini_batch)
                image_embs = image_outputs.pooler_output  # Compute mean embeddings for the mini-batch
                image_embs_list.append(image_embs)

        # Concatenate all mini-batch embeddings into a single tensor
        image_embs = torch.cat(image_embs_list, dim=0)
        image_embs = F.normalize(image_embs, p=2, dim=-1).detach().cpu().numpy()

        # Process each question embedding and record metrics
        for question_emb, labels, doc_type, label_types in zip(batch['question_embs'], batch['labels'], batch['doc_types'], batch['label_types']):
            df_index = len(self.evaluation_df)
            probs = np.dot(question_emb, image_embs.T)
            self.evaluation_df.loc[df_index, 'dataset'] = batch['dataset']
            self.evaluation_df.loc[df_index, 'doc_id'] = batch['doc_id']
            self.evaluation_df.loc[df_index, 'probs'] = probs
            self.evaluation_df.loc[df_index, 'labels'] = labels
            self.evaluation_df.loc[df_index, 'doc_type'] = doc_type
            self.evaluation_df.loc[df_index, 'label_types'] = label_types
        return

    def on_validation_epoch_end(self):
        # Initialize metrics storage
        dataset_metrics = {}
        total_recall1 = []
        total_ndcg = []

        # Iterate over unique datasets
        for dataset in self.evaluation_df['dataset'].unique():
            df_filtered = self.evaluation_df[self.evaluation_df['dataset'] == dataset]

            # Initialize metric lists
            single_recall1_list = []
            single_recall3_list = []
            single_recall5_list = []
            multi_ndcg_list = []

            # Iterate over each row (sample)
            for index, row in df_filtered.iterrows():
                probs = row['probs']   # Model predictions (probabilities)
                labels = row['labels']  # Ground truth labels

                # Handle single-label case (len(labels) == 1)
                if len(labels) == 1:
                    # Calculate Recall@1, Recall@3, and Recall@5
                    top_indices = np.argsort(-probs)[:5]  # Get top 5 predictions
                    single_recall1 = int(labels[0] == top_indices[0])  # Recall@1
                    single_recall3 = int(labels[0] in top_indices[:3])  # Recall@3
                    single_recall5 = int(labels[0] in top_indices[:5])  # Recall@5

                    single_recall1_list.append(single_recall1)
                    single_recall3_list.append(single_recall3)
                    single_recall5_list.append(single_recall5)
                    total_recall1.append(single_recall1)  # Accumulate for total Recall@1

                # Handle multi-label case (len(labels) > 1)
                else:
                    # Create relevance vector for NDCG
                    relevance = np.zeros(len(probs))
                    relevance[labels] = 1  # Assume relevance of 1 for relevant items

                    # Calculate NDCG
                    ndcg = ndcg_score([relevance], [probs])
                    multi_ndcg_list.append(ndcg)
                    total_ndcg.append(ndcg)  # Accumulate for total NDCG

            # Store dataset-level metrics
            dataset_metrics[dataset] = {
                'recall@1': np.mean(single_recall1_list) if single_recall1_list else None,
                'recall@3': np.mean(single_recall3_list) if single_recall3_list else None,
                'recall@5': np.mean(single_recall5_list) if single_recall5_list else None,
                'NDCG': np.mean(multi_ndcg_list) if multi_ndcg_list else None
            }

            # Log dataset-specific metrics
            self.log(f'{dataset} Recall@1 (single-label)', dataset_metrics[dataset]['recall@1'])
            self.log(f'{dataset} Recall@3 (single-label)', dataset_metrics[dataset]['recall@3'])
            self.log(f'{dataset} Recall@5 (single-label)', dataset_metrics[dataset]['recall@5'])
            if dataset_metrics[dataset]['NDCG'] is not None:
                self.log(f'{dataset} NDCG (multi-label)', dataset_metrics[dataset]['NDCG'])

        # Compute total metrics across all datasets
        total_recall1_mean = np.mean(total_recall1) if total_recall1 else None
        total_ndcg_mean = np.mean(total_ndcg) if total_ndcg else None

        # Log total metrics
        self.log('recall_at_1', total_recall1_mean)
        self.log('ndcg', total_ndcg_mean)

        # Reset the evaluation DataFrame for the next epoch
        self.evaluation_df = pd.DataFrame(columns=['dataset', 'doc_id', 'probs', 'labels', 'doc_type', 'label_types'])
        torch.cuda.empty_cache()
        gc.collect()
        return


In [16]:
lit_model = LitDocEmbModel(image_model_name, text_model_name, mini_batch_size=32)

In [17]:
checkpoint_callback = ModelCheckpoint(
    monitor='recall_at_1',
    verbose=True,
    save_top_k=1,
    mode='max',
    dirpath='checkpoint',
    filename=f"{text_model_name.split('/')[-1]}-{image_model_name.split('/')[-1]}-finetune-batch={BATCH_SIZE}-BSM"+"-{epoch:2d}-{recall_at_1:.4f}"
)

earlystop_callback = EarlyStopping(
    monitor="recall_at_1", 
    patience=6, 
    verbose=True,
    mode="max"
)

In [18]:
trainer = L.Trainer(
    max_epochs=1000, 
    precision=16, 
    callbacks=[checkpoint_callback, earlystop_callback],
    val_check_interval=0.5
)

C:\Users\dust\anaconda3\envs\py311_torch2\Lib\site-packages\lightning\fabric\connector.py:571: `precision=16` is supported for historical reasons but its usage is discouraged. Please set your precision to 16-mixed instead!
Using 16bit Automatic Mixed Precision (AMP)
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs


In [ ]:
trainer.fit(lit_model, train_dataloader, eval_dataloader)

C:\Users\dust\anaconda3\envs\py311_torch2\Lib\site-packages\lightning\pytorch\callbacks\model_checkpoint.py:654: Checkpoint directory C:\Users\dust\Documents\DoClip\checkpoint exists and is not empty.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name        | Type              | Params | Mode 
----------------------------------------------------------
0 | image_model | BeitModel         | 85.8 M | train
1 | loss        | BinarySigmoidLoss | 2      | train
----------------------------------------------------------
85.8 M    Trainable params
0         Non-trainable params
85.8 M    Total params
343.231   Total estimated model params size (MB)
239       Modules in train mode
0         Modules in eval mode


Sanity Checking: |                                                                               | 0/? [00:00<…

C:\Users\dust\anaconda3\envs\py311_torch2\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:424: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.
C:\Users\dust\.cache\huggingface\modules\transformers_modules\Alibaba-NLP\new-impl\40ced75c3017eb27626c9d4ea981bde21a2662f4\modeling.py:579: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(
C:\Users\dust\anaconda3\envs\py311_torch2\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:424: The 'train_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_

Training: |                                                                                      | 0/? [00:00<…

Validation: |                                                                                    | 0/? [00:00<…

...

Metric recall_at_1 improved by 0.001 >= min_delta = 0.0. New best score: 0.303
Epoch 5, global step 10805: 'recall_at_1' reached 0.30257 (best 0.30257), saving model to 'C:\\Users\\dust\\Documents\\DoClip\\checkpoint\\gte-base-en-v1.5-dit-base-finetuned-rvlcdip-finetune-batch=512-BSM-epoch= 5-recall_at_1=0.3026.ckpt' as top 1


Validation: |                                                                                    | 0/? [00:00<…

Epoch 6, global step 11706: 'recall_at_1' was not in top 1


Validation: |                                                                                    | 0/? [00:00<…

Epoch 6, global step 12606: 'recall_at_1' was not in top 1


In [21]:
trainer.validate(lit_model, eval_dataloader)

LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
C:\Users\dust\anaconda3\envs\py311_torch2\Lib\site-packages\lightning\pytorch\trainer\connectors\data_connector.py:424: The 'val_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=11` in the `DataLoader` to improve performance.


Validation: |                                                                                    | 0/? [00:00<…

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃             Validate metric             ┃              DataLoader 0               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│   MMLongBench-Doc NDCG (multi-label)    │           0.3902868926525116            │
│ MMLongBench-Doc Recall@1 (single-label) │           0.03711790218949318           │
│ MMLongBench-Doc Recall@3 (single-label) │           0.0786026194691658            │
│ MMLongBench-Doc Recall@5 (single-label) │           0.14192140102386475           │
│    MP-DOCVQA Recall@1 (single-label)    │           0.2255639135837555            │
│    MP-DOCVQA Recall@3 (single-label)    │           0.4266435205936432            │
│    MP-DOCVQA Recall@5 (single-label)    │           0.5270869731903076            │
│                  ndcg                   │           0.3902868926525116            │
│               recall_at_1               │           0.2102745771408081            │
└─────────────────────────────────────────┴─────────────────────────────────────────┘

[{'MMLongBench-Doc Recall@1 (single-label)': 0.03711790218949318,
  'MMLongBench-Doc Recall@3 (single-label)': 0.0786026194691658,
  'MMLongBench-Doc Recall@5 (single-label)': 0.14192140102386475,
  'MMLongBench-Doc NDCG (multi-label)': 0.3902868926525116,
  'MP-DOCVQA Recall@1 (single-label)': 0.2255639135837555,
  'MP-DOCVQA Recall@3 (single-label)': 0.4266435205936432,
  'MP-DOCVQA Recall@5 (single-label)': 0.5270869731903076,
  'recall_at_1': 0.2102745771408081,
  'ndcg': 0.3902868926525116}]